# F1 de identificación FDI de la etapa 1 con IoU > 0,5 (sin GPU)
Especificación: `claude/encargo_f1_iou05.md`. **Criterio principal nuevo:** un diente de la referencia está identificado si la máscara predicha del mismo número FDI tiene **IoU > 0,5** con la suya (convención de panoptic quality; RQ = F1). El criterio anterior (Dice ≥ 0,5, equivalente a IoU ≥ 1/3) queda como sensibilidad.

**Input:** `instancias.pkl.gz` de `map-segmentadores` (matriz de confusión de píxeles por foto y modelo; YOLO en su versión pintada a conf ≥ 0,25, la del benchmark), manifiesto de `datasets-limpios-icdas-segmentacion` y CSV del benchmark (`benchmark-segmentacion-cv-nino`). Sin re-inferir.

Estructura idéntica al benchmark (`nb_train_seg.ipynb`): por foto y clase se consideran las clases presentes en la referencia o predichas con ≥ 50 px; sensibilidad = dientes de referencia identificados / dientes de referencia; VPP = clases predichas (≥ 50 px) que cumplen el criterio / clases predichas; F1 = 2·Se·VPP/(Se+VPP). IC 95 % por bootstrap de niños (1.000 remuestreos, `default_rng(0)`, igual que `metricas_ic95.csv`), comparación contra el mejor con Holm, subgrupos por vista y retractor.

In [ ]:
import os, glob, gzip, pickle, time
import numpy as np, pandas as pd
IN, W = '/kaggle/input', '/kaggle/working'
OUT = f'{W}/f1'; os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 250)
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
INST = pickle.load(gzip.open(find('instancias.pkl.gz')))
man = pd.read_csv(find('segmentation_manifest.csv'))
RES = os.path.dirname(find('metricas_ic95.csv'))
REF = pd.read_csv(f'{RES}/metricas_ic95.csv').set_index('model')
MODELS = ['unetpp_r50', 'deeplabv3p_r50', 'segformer_b2', 'yolo11m_seg']
LABEL = {'unetpp_r50': 'U-Net++', 'deeplabv3p_r50': 'DeepLabV3+', 'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg'}
for m in MODELS:
    assert len(INST[m]) == len(man) and all('cm' in r for r in INST[m].values()), f'falta la matriz de confusión de {m}'
NC = next(iter(INST[MODELS[0]].values()))['cm'].shape[0]
# FDI de cada índice de clase (labelmap Universal, fondo = 0), igual que en el benchmark
def to_fdi(u):
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
# el orden de índices del labelmap se verifica contra las clases FDI guardadas en las instancias de referencia del caché no es posible (solo cm);
# se lee el labelmap del zip para no suponer el orden
import zipfile, re
z = zipfile.ZipFile(find('fdi_segmentation_curated.zip'))
lm = [l.strip() for l in z.read([n for n in z.namelist() if n.endswith('labelmap.txt')][0]).decode().splitlines() if l.strip() and not l.startswith('#')]
NAMES = [l.split(':')[0].strip() for l in lm]
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0: NAMES.insert(0, NAMES.pop(bg[0]))
FDI = [None] + [to_fdi(int(re.search(r'\d+', n).group())) for n in NAMES[1:]]
assert NC == 33 and len(FDI) == NC
print('instancias del caché:', {LABEL[m]: len(INST[m]) for m in MODELS}, '| clases', NC, '| columnas del manifiesto:', list(man.columns))
print(REF[['dice_tooth', 'id_sens', 'id_ppv', 'id_f1', 'numbering_acc']].round(4).to_string())

## Tabla por foto y clase (desde la matriz de confusión de píxeles)
`inter = cm[c,c]`, `ng = cm[c,:].sum()`, `npx = cm[:,c].sum()`; Dice = 2·inter/(ng+npx); IoU = inter/(ng+npx−inter). Se guarda también la exactitud de numeración del benchmark (diente de referencia cubierto ≥ 50 % por píxeles de diente predichos; correcto si la clase mayoritaria es la suya), como control.

In [ ]:
rows = []
for m in MODELS:
    for j, f, child, view, cond in zip(man.image_id, man.fold, man.child_id, man.view, man.condition):
        cm = INST[m][j]['cm'].astype(np.int64); ng = cm.sum(1); npx = cm.sum(0); inter = np.diag(cm)
        for c in range(1, NC):
            if ng[c] == 0 and npx[c] < 50: continue
            cov = ng[c] > 0 and cm[c, 1:].sum() / ng[c] >= 0.5
            rows.append((m, j, f, child, view, cond, FDI[c], ng[c], npx[c], inter[c],
                         2 * inter[c] / max(1, ng[c] + npx[c]), inter[c] / max(1, ng[c] + npx[c] - inter[c]),
                         cov, cov and (cm[c, 1:].argmax() + 1 == c)))
T = pd.DataFrame(rows, columns=['model', 'image_id', 'fold', 'child_id', 'view', 'condition', 'tooth', 'gt_px', 'pred_px', 'inter', 'dice', 'iou', 'covered', 'numbered_ok'])
T['in_gt'] = T.gt_px > 0; T['in_pred'] = T.pred_px >= 50
CRIT = {'dice05': lambda d: d.dice >= 0.5, 'iou05': lambda d: d.iou > 0.5, 'iou075': lambda d: d.iou > 0.75}
CRIT_LABEL = {'dice05': 'Dice ≥ 0,5 (anterior)', 'iou05': 'IoU > 0,5 (principal)', 'iou075': 'IoU > 0,75'}
for k, f in CRIT.items():
    ok = f(T); T[f'id_{k}'] = T.in_gt & ok; T[f'tp_{k}'] = T.in_pred & ok
print(len(T), 'filas foto×clase |', T[T.model == MODELS[0]].in_gt.sum(), 'dientes de referencia por modelo')

## Chequeo previo: con Dice ≥ 0,5 se debe reproducir el F1 de `metricas_ic95.csv` (±0,001)

In [ ]:
def agg(d, k):
    g, p = d[d.in_gt], d[d.in_pred]
    se = g[f'id_{k}'].mean(); pp = p[f'tp_{k}'].mean() if len(p) else np.nan
    return dict(sens=se, ppv=pp, f1=2 * se * pp / (se + pp) if se + pp > 0 else 0.0)
chk = []
for m in MODELS:
    a = agg(T[T.model == m], 'dice05'); g = T[(T.model == m) & T.in_gt]
    chk.append(dict(modelo=LABEL[m], f1=a['f1'], f1_ref=REF.loc[m, 'id_f1'], sens=a['sens'], sens_ref=REF.loc[m, 'id_sens'], ppv=a['ppv'], ppv_ref=REF.loc[m, 'id_ppv'],
                    numeracion=g[g.covered].numbered_ok.mean(), numeracion_ref=REF.loc[m, 'numbering_acc']))
chk = pd.DataFrame(chk); chk['ok'] = (chk.f1 - chk.f1_ref).abs() <= 0.001
chk.to_csv(f'{OUT}/chequeo_previo.csv', index=False); print(chk.round(4).to_string(index=False))
assert chk.ok.all(), 'EL F1 CON DICE >= 0,5 NO REPRODUCE metricas_ic95.csv: se detiene'
print('chequeo OK')

## Principal (IoU > 0,5) y sensibilidad del umbral: IC 95 % por bootstrap de niños, media ± DE por fold
Mismo bootstrap que el benchmark: `rng = default_rng(0)`, 1.000 remuestreos de los 96 niños con reemplazo; los mismos remuestreos para todos los modelos y criterios. Se calcula con sumas por niño (equivale a concatenar las filas de los niños remuestreados).

In [ ]:
KIDS = np.array(sorted(man.child_id.unique())); rng = np.random.default_rng(0); B = 1000
draws = [rng.choice(KIDS, len(KIDS)) for _ in range(B)]
KI = {c: i for i, c in enumerate(KIDS)}
WK = np.stack([np.bincount([KI[c] for c in d], minlength=len(KIDS)) for d in draws])   # (B, niños)
def boot(d, k):
    s = d.groupby('child_id').agg(ng=('in_gt', 'sum'), idn=(f'id_{k}', 'sum'), npd=('in_pred', 'sum'), tp=(f'tp_{k}', 'sum')).reindex(KIDS, fill_value=0)
    se = WK @ s.idn.values / (WK @ s.ng.values); pp = WK @ s.tp.values / (WK @ s.npd.values)
    return pd.DataFrame(dict(sens=se, ppv=pp, f1=2 * se * pp / (se + pp)))
BOOT, res, fold_rows = {}, [], []
for k in CRIT:
    for m in MODELS:
        d = T[T.model == m]; pt = agg(d, k); bs = boot(d, k); BOOT[(k, m)] = bs
        pf = pd.DataFrame([dict(fold=f, **agg(x, k)) for f, x in d.groupby('fold')]); fold_rows += [dict(criterio=k, modelo=LABEL[m], **r) for r in pf.to_dict('records')]
        res.append(dict(criterio=k, modelo=LABEL[m], **pt, **{f'{c}_lo': bs[c].quantile(.025) for c in bs}, **{f'{c}_hi': bs[c].quantile(.975) for c in bs},
                        **{f'{c}_fold_media': pf[c].mean() for c in ('sens', 'ppv', 'f1')}, **{f'{c}_fold_de': pf[c].std() for c in ('sens', 'ppv', 'f1')}))
res = pd.DataFrame(res); pd.DataFrame(fold_rows).to_csv(f'{OUT}/f1_por_fold.csv', index=False)
res[res.criterio == 'iou05'].drop(columns='criterio').to_csv(f'{OUT}/f1_iou05_resumen.csv', index=False)
for k in CRIT:
    print(f'\n{CRIT_LABEL[k]}: combinado [IC 95 %] | por fold media ± DE')
    for _, r in res[res.criterio == k].iterrows():
        print(f"{r.modelo:13s} " + '  '.join(f"{c} {r[c]:.3f} [{r[c + '_lo']:.3f}–{r[c + '_hi']:.3f}]" for c in ('sens', 'ppv', 'f1')) + f"  | F1 {r.f1_fold_media:.3f} ± {r.f1_fold_de:.3f}")
if 'id_f1_lo' in REF: print('\nIC del benchmark (Dice ≥ 0,5) para comparar con el de arriba:', {LABEL[m]: (round(REF.loc[m, 'id_f1_lo'], 3), round(REF.loc[m, 'id_f1_hi'], 3)) for m in MODELS})
sen = res.pivot(index='modelo', columns='criterio', values='f1')[list(CRIT)].reindex([LABEL[m] for m in MODELS])
rk = sen.rank(ascending=False).astype(int).add_prefix('orden_')
sen = pd.concat([sen, rk], axis=1); sen.to_csv(f'{OUT}/f1_umbral_sensibilidad.csv')
print('\nF1 según el criterio (y orden de los modelos):'); print(sen.round(3).to_string())

## Comparación pareada contra el mejor (bootstrap pareado de niños, Holm), para cada criterio

In [ ]:
cmp = []
for k in CRIT:
    r = res[res.criterio == k].sort_values('f1', ascending=False); top = [m for m in MODELS if LABEL[m] == r.modelo.iloc[0]][0]; part = []
    for m in MODELS:
        if m == top: continue
        dl = BOOT[(k, top)].f1.values - BOOT[(k, m)].f1.values
        part.append(dict(criterio=k, mejor=LABEL[top], vs=LABEL[m], dif=agg(T[T.model == top], k)['f1'] - agg(T[T.model == m], k)['f1'],
                         lo=np.quantile(dl, .025), hi=np.quantile(dl, .975), p=min(1, 2 * min((dl <= 0).mean(), (dl >= 0).mean()))))
    part = pd.DataFrame(part).sort_values('p').reset_index(drop=True); n = len(part)
    part['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(part.p)])); cmp.append(part)
cmp = pd.concat(cmp); cmp[cmp.criterio == 'iou05'].to_csv(f'{OUT}/f1_iou05_pareada.csv', index=False); cmp.to_csv(f'{OUT}/f1_pareada_todos_criterios.csv', index=False)
print(cmp.round(3).to_string(index=False))

## Subgrupos (vista y retractor) y por diente, con IoU > 0,5 (y Dice ≥ 0,5 para comparar)

In [ ]:
sub = pd.DataFrame([dict(modelo=LABEL[m], grupo=f'{c}={v}', n_img=d.image_id.nunique(), n_dientes=int(d.in_gt.sum()),
                         **{f'{x}_{k}': y for k in ('iou05', 'dice05') for x, y in agg(d, k).items()})
                    for c in ('view', 'condition') for (m, v), d in T.groupby(['model', c])])
sub.to_csv(f'{OUT}/f1_iou05_subgrupos.csv', index=False)
print(sub.pivot(index='grupo', columns='modelo', values='f1_iou05').round(3).to_string())
print(sub.groupby('grupo').n_img.first().to_string())
pt = []
for (m, t), d in T.groupby(['model', 'tooth']):
    g, p = d[d.in_gt], d[d.in_pred]
    pt.append(dict(modelo=LABEL[m], diente=t, n_ref=len(g), n_pred=len(p), sens_iou05=g.id_iou05.mean(), ppv_iou05=p.tp_iou05.mean() if len(p) else np.nan,
                   sens_dice05=g.id_dice05.mean(), ppv_dice05=p.tp_dice05.mean() if len(p) else np.nan, iou_mediana=g.iou.median()))
pt = pd.DataFrame(pt); pt.to_csv(f'{OUT}/f1_iou05_por_diente.csv', index=False)
print(pt.pivot(index=['diente'], columns='modelo', values='sens_iou05').round(3).assign(n_ref=pt.groupby('diente').n_ref.first()).to_string())

## Franja 1/3 ≤ IoU ≤ 0,5: dientes que cuentan con el criterio anterior y no con el nuevo
Dice ≥ 0,5 equivale a IoU ≥ 1/3. Se cuentan dientes de referencia (lado de la sensibilidad) y clases predichas (lado del VPP).

In [ ]:
T['band_ref'] = T.in_gt & T.id_dice05 & ~T.id_iou05; T['band_pred'] = T.in_pred & T.tp_dice05 & ~T.tp_iou05
band = T.groupby('model').agg(dientes_ref=('in_gt', 'sum'), franja_ref=('band_ref', 'sum'), predichas=('in_pred', 'sum'), franja_pred=('band_pred', 'sum')).reindex(MODELS)
band['pct_ref'] = band.franja_ref / band.dientes_ref; band.index = [LABEL[m] for m in band.index]
print(band.round(4).to_string())
grp = lambda t: 'terceros molares' if t % 10 == 8 else 'molares 1.º–2.º' if t % 10 in (6, 7) else 'premolares' if t % 10 in (4, 5) else 'incisivos y caninos'
bt = T[T.band_ref].assign(grupo=lambda d: d.tooth.map(grp))
by_grp = pd.crosstab(bt.grupo, bt.model.map(LABEL), margins=True)
ref_grp = T[T.in_gt & (T.model == MODELS[0])].tooth.map(grp).value_counts()
by_grp['n_ref (por modelo)'] = ref_grp; print(by_grp.to_string())
by_tooth = pd.crosstab(bt.tooth, bt.model.map(LABEL)); by_tooth['total'] = by_tooth.sum(1)
print(by_tooth.sort_values('total', ascending=False).head(12).to_string())
by_view = pd.crosstab(bt.view, bt.model.map(LABEL)); print(by_view.to_string())
pd.concat({'por_grupo': by_grp, 'por_diente': by_tooth}, names=['tabla']).to_csv(f'{OUT}/franja_033_05.csv')

## Exactitud de numeración (control): no depende del umbral de Dice
En el benchmark: entre los dientes de referencia cubiertos ≥ 50 % por píxeles de diente predichos, fracción cuya clase predicha mayoritaria es la correcta. Se informa también restringida a los dientes identificados con IoU > 0,5, solo como referencia.

In [ ]:
num = []
for m in MODELS:
    g = T[(T.model == m) & T.in_gt]
    num.append(dict(modelo=LABEL[m], numeracion=g[g.covered].numbered_ok.mean(), n_cubiertos=int(g.covered.sum()), numeracion_ref=REF.loc[m, 'numbering_acc'],
                    cubiertos_con_iou05=g[g.covered].id_iou05.mean()))
num = pd.DataFrame(num); num.to_csv(f'{OUT}/numeracion_control.csv', index=False); print(num.round(4).to_string(index=False))
print(sorted(os.listdir(OUT)))

## "Diente encontrado con su número FDI" de la evaluación encadenada (pipeline-icdas-v4), con IoU > 0,5
En `pipeline-icdas-v4`, un recorte cuenta como encontrado si la máscara predicha por SegFormer-B2 (modelo del fold en que el niño fue prueba) tiene **algún píxel** de la clase FDI del recorte en esa foto (`pfdib.notna()`): sin umbral de Dice ni de IoU ni área mínima. Aquí se reproduce esa cifra con las matrices de confusión del caché (mismas predicciones fuera de fold) y se recalcula exigiendo IoU > 0,5 (y Dice ≥ 0,5) entre la máscara predicha y la de referencia de ese diente.

In [ ]:
zp = find('icdas_v4.zip')
if zp is None: print('falta base-icdas-v4 como input: se omite esta celda')
else:
    zi = zipfile.ZipFile(zp); mi = pd.read_csv(zi.open([n for n in zi.namelist() if n.endswith('manifest.csv')][0]), dtype={'micdas': str})
    ev = mi[mi.seg_mismo_fold.fillna(False).astype(bool) & mi.fdi.notna()].copy()
    key = {str(j): j for j in man.image_id}; IDX = {f: c for c, f in enumerate(FDI) if f is not None}
    out = []
    for r in ev.itertuples():
        j = key.get(str(r.seg_image_id)) or key.get(str(int(float(r.seg_image_id))))
        cm = INST['segformer_b2'][j]['cm'].astype(np.int64); c = IDX[int(r.fdi)]
        ng, npx, it = cm[c].sum(), cm[:, c].sum(), cm[c, c]
        out.append(dict(micdas=r.micdas, view=r.view, encontrado=npx > 0, iou=it / max(1, ng + npx - it), dice=2 * it / max(1, ng + npx)))
    E = pd.DataFrame(out); E['iou05'] = E.iou > 0.5; E['dice05'] = E.dice >= 0.5
    print(len(E), 'recortes |', ev.child_id.nunique(), 'niños |', ev.seg_image_id.nunique(), 'fotos')
    tab = pd.concat([E[['encontrado', 'dice05', 'iou05']].mean().rename('todos'), E.groupby('micdas')[['encontrado', 'dice05', 'iou05']].mean().T], axis=1)
    tab.loc['n'] = [len(E)] + E.micdas.value_counts().reindex(tab.columns[1:]).tolist()
    print(tab.round(4).to_string()); print('IoU mediana del diente (máscara predicha vs referencia):', round(E.iou.median(), 3))
    tab.to_csv(f'{OUT}/encadenada_encontrado_iou05.csv')